In [1]:
include("../src/main.jl");

In [2]:
dump_filepath = "../src/models/iharm3dDumps/dump_001.h5";
#dump_filepath = "../../../../Downloads/torus.out0.00356.h5";

In [3]:
#TODO: put this in reading file
const N1 = 128
const N2 = 64
const N3 = 32

const METRIC = "FMKS" #FMKS or MKS TODO: prob have to be read from file
const trat_large = 20. #TODO: prob have to be read from file
const trat_small = 1. #TODO: prob have to be read from file
const beta_crit = 1.0 #TODO: prob have to be read from file
const game = (4. /3.)  # Ion adiabatic index  TODO: prob have to be read from file
const gamp = (5. /3.)  # Electron adiabatic index TODO: prob have to be read from file
const gam = (13. /9.)  # Total adiabatic index TODO: prob have to be read from file
const Ne_factor = 1.0  # Scaling factor for electron number density TODO: prob have to be read from file
const rmin_geo = 1.00187575798832   #TODO: Has to be read from file as Rin and compared to the value chosen
const rmax_geo = 100. #TODO: Has to be read from file as Rin and compared to the value chosen
const th_beg = 1.74e-2 #TODO: Idk where this comes from, check ipole source code
const sigma_cut = 1.0 #TODO: maybe put it somewhere else?
const sigma_cut_high = -1.0
const startx::MVec4 = [0, 1.874000951149813e-03, 0, 0]#TODO: prob have to be read from file
const stopx::MVec4 = [1, 6.907755278982138e+00, 1, 2 * π]#TODO: prob have to be read from file
const dx::MVec4 =[0, 5.395219748461709e-02, 1.562500000000000e-02, 1.963495408493621e-01]
const bhspin = 0.9375 #TODO: prob have to be read from file
const hslope = 0.3 #TODO: prob have to be read from file


0.3

In [4]:
simulation_data = load_data(dump_filepath);

Loading data from '../src/models/iharm3dDumps/dump_001.h5' into 'iharm' module...
Primitives successfully loaded. Dimensions: (128, 64, 32)
Calculating physical quantities...
Using mixed tp_over_te with trat_small = 1.0, trat_large = 20.0, and beta_crit = 1.0
All primitives successfully loaded. Dimensions: (128, 64, 32)


In [9]:
#Analytic parameters

#Setting up the parameters
#Observer distance in Rg
const ro = 1000.0
#Observer inclination in degrees
const th = 80.0

#Observer azimuth in degrees
const phi = 0.0

# Number of pixels in the x and y direction. The number of geodesics calculated will be res^2
const res = 160
const pixels_x = 160
const pixels_y = 160
# Distance to the source in parsecs
const SourceD = 16.9e6 * PC
const Rout = 1000.0
const Rstop = 100.0
const Rh = 1 + sqrt(1. - bhspin * bhspin);

#Check if these are correct
#const cstartx = [0.0, log(Rh), 0.0, 0.0]#TODO: prob have to be read from file
const cstartx = MVec4(0.0, log(Rh), 0.0, 0.0)#TODO: prob have to be read from file
const cstopx = MVec4(0.0, log(Rout), 1.0, 2.0 * π)#TODO: prob have to be read from file

# Frequency observed by the camera in Hz
const freq = 230e9;

# Size of the screen in Rg in both directions
const DXsize = SourceD/L_unit/MUAS_PER_RAD * 160
const DYsize = SourceD/L_unit/MUAS_PER_RAD * 160
# Observer fov in radians (this can be understood as size of the plane camera sees over the distance ro)
# This should be atan, but for small angles it is approximately equal to the angle itself
const fovx = DXsize/ro
const fovy = DYsize/ro
const xoff = 0.0
const yoff = 0.0


0.0

In [10]:
include("../src/main.jl");
# Find camera in native coordinates
Xcamera = MVec4(camera_position(ro, th, phi, bhspin, Rout))

# Scales the intensity of each pixel by the real size of each pixel
scale_factor = CalculateScaleFactor(DXsize, DYsize, pixels_x, pixels_y, SourceD, L_unit)
println("scale_factor = $scale_factor")
const maxnstep = 15000
# Generate geodesics
println("Utilizing $(Threads.nthreads()) threads for geodesic calculation.")
freq_unitless = freq * HPL/(ME * CL * CL)  # Convert frequency to unitless
Image = zeros(Float64, pixels_x, pixels_y)
for i in 0:(pixels_x - 1)
    println("Processing row $i out of $(pixels_x)")

    Threads.@threads for j in 0:(pixels_y - 1)
        traj = Vector{OfTraj}()
        sizehint!(traj, maxnstep)
        nstep = get_pixel(traj, i, j, Xcamera, maxnstep, fovx, fovy, freq_unitless, pixels_x, pixels_y, bhspin, Rh, Rout, Rstop, xoff, yoff)

        resize!(traj, length(traj))
        integrate_emission!(traj, length(traj), Image, i + 1, j + 1, freq, bhspin, simulation_data)
    end
end
Image *= freq^3;

scale_factor = 2.350438638182906
Utilizing 8 threads for geodesic calculation.
Processing row 0 out of 160
Processing row 1 out of 160
Processing row 2 out of 160
Processing row 3 out of 160
Processing row 4 out of 160
Processing row 5 out of 160
Processing row 6 out of 160
Processing row 7 out of 160
Processing row 8 out of 160
Processing row 9 out of 160
Processing row 10 out of 160
Processing row 11 out of 160
Processing row 12 out of 160
Processing row 13 out of 160
Processing row 14 out of 160
Processing row 15 out of 160
Processing row 16 out of 160
Processing row 17 out of 160
Processing row 18 out of 160
Processing row 19 out of 160
Processing row 20 out of 160
Processing row 21 out of 160
Processing row 22 out of 160
Processing row 23 out of 160
Processing row 24 out of 160
Processing row 25 out of 160
Processing row 26 out of 160
Processing row 27 out of 160
Processing row 28 out of 160
Processing row 29 out of 160
Processing row 30 out of 160
Processing row 31 out of 160
Pro

In [7]:
OutputStokesParameters(Image, freq, scale_factor, res, SourceD)

Image processing complete. Calculating total flux and averages...
Scale = 2.350438638182906e+00
imax = 27, jmax = 89, Imax = 0.006416052023077334, Iavg = 0.0014917239582839844
Using freq_cgs = 2.3e11, Ftot = 89.75886410377706
nuLnu = 7.054880559756697e42


In [11]:
using DelimitedFiles

writedlm("./Image.txt", Image)


In [29]:
using DelimitedFiles

Image_loaded = readdlm("./Image.txt")


40×40 Matrix{Float64}:
 8.23342e-5   8.52182e-5   9.06941e-5   …  0.00202057   0.00168526
 7.75398e-5   8.27666e-5   9.20324e-5      0.000788816  0.000684682
 8.79549e-5   9.5871e-5    0.000104739     0.000837101  0.000739291
 0.000101976  0.000111416  0.000122622     0.000991732  0.000861424
 0.000114065  0.00012357   0.00013348      0.00117229   0.00103081
 0.000124716  0.000130253  0.000143087  …  0.00133275   0.00120975
 0.000128215  0.000129155  0.000142148     0.00138372   0.00128731
 0.000120381  0.000137384  0.000163254     0.0014374    0.00131772
 0.000135014  0.000164139  0.000190291     0.00145569   0.00134545
 0.000163165  0.000191066  0.000215823     0.00124681   0.00132045
 0.000187987  0.000212363  0.000236721  …  0.00117972   0.00114855
 0.000206514  0.000231142  0.000256864     0.00114219   0.00108454
 0.000228709  0.000240015  0.000286118     0.00112855   0.00104018
 ⋮                                      ⋱               
 0.000278596  0.00026986   0.000270525     0.0

In [15]:
Image[13,21]*freq^3

0.0022925422377131936

In [6]:
# From Ipole

In [12]:
using HDF5

function load_ipole_unpol(fname)
    hfp = h5open(fname, "r") do file
        # Read values
        dx = read(file["header/camera/dx"])
        dsource = read(file["header/dsource"])
        lunit = read(file["header/units/L_unit"])
        scale = read(file["header/scale"])

        fov_muas = dx / dsource * lunit * 2.06265e11

        # evpa_0 might not exist
        evpa_0 = haskey(file["header"], "evpa_0") ? read(file["header/evpa_0"]) : "W"

        # Load and transpose unpol
        unpol = read(file["unpol"])
        unpol_t = permutedims(unpol, (2,1))

        return unpol_t, fov_muas, scale, evpa_0
    end

    return hfp
end

load_ipole_unpol (generic function with 1 method)

In [13]:
Image1, _, _, _= load_ipole_unpol("../../ipole/image.h5")

([8.233421941705342e-5 8.521815581569922e-5 … 0.0020205738150697056 0.0016852612368673543; 7.753977831356059e-5 8.276657275500758e-5 … 0.000788815964249771 0.0006846818425350721; … ; 0.00011429092296199376 0.00011686607621426469 … 0.0014757403048885089 0.0013350815276502342; 0.00011956884484868792 0.0001228642228144521 … 0.0010126581520189663 0.0008398516209955787], 160.0, 37.6070182109265, "N")

In [29]:
include("../src/main.jl");

Xmvec4 = MVec4(0.0, 6.907755278982137, 0.05322792008519173, 0.0)
lconn_analytic = Tensor3D(undef)
fill!(lconn_analytic, 0.0)

spin = 0.9375
using BenchmarkTools
@btime get_connection_analytic!($Xmvec4, $lconn_analytic, $bhspin)

  3.584 μs (289 allocations: 4.53 KiB)


-8.013857346109405e-5

In [50]:
include("../src/main.jl");
lconn_FD = Tensor3D(undef)
fill!(lconn_FD, 0.0)
Xmvec4 = MVec4(0.0, 6.907755278982137, 0.05322792008519173, 0.0)
get_connection(Xmvec4, bhspin, lconn_FD)
using BenchmarkTools
@btime get_connection($Xmvec4, $bhspin, $lconn_FD)


  292.820 μs (18983 allocations: 315.45 KiB)


4×4×4 MArray{Tuple{4, 4, 4}, Float64, 3, 64} with indices SOneTo(4)×SOneTo(4)×SOneTo(4):
[:, :, 1] =
 1.99999e-9    0.001002    -2.49783e-9   -1.60277e-10
 9.97998e-10  -1.99992e-9   2.68274e-16  -7.99782e-11
 1.38473e-12  -2.8685e-12   3.72255e-19   9.95299e-11
 9.37497e-13   9.8227e-10  -3.22668e-8   -7.51297e-14

[:, :, 2] =
  0.001002     2.00189       0.0768187  -8.02987e-5
 -1.99992e-9   0.998945      0.0383326   8.01387e-5
 -2.8685e-12  -0.000695259   0.996018    5.00313e-5
  9.8227e-10   4.57504e-5   -0.0161297   0.97612

[:, :, 3] =
 -2.49783e-9    0.0768187  -55.3629      2.07213e-10
  2.68274e-16   0.0383326  -27.6261     -1.56255e-14
  3.72255e-19   0.996018    -1.82408    -2.16819e-17
 -3.22668e-8   -0.0161297   -0.0259514  17.209

[:, :, 4] =
 -1.60277e-10  -8.02987e-5   2.07213e-10  -0.170962
 -7.99782e-11   8.01387e-5  -1.56255e-14  -0.0853103
  9.95299e-11   5.00313e-5  -2.16819e-17  -0.0532603
 -7.51297e-14   0.97612     17.209        -8.01386e-5

In [49]:
include("../src/main.jl");
lconn_FD = Tensor3D(undef)
fill!(lconn_FD, 0.0)
Xmvec4 = MVec4(0.0, 6.907755278982137, 0.05322792008519173, 0.0)
get_connection(Xmvec4, bhspin, lconn_FD)
using BenchmarkTools
@btime get_connection($Xmvec4, $bhspin, $lconn_FD)
#@btime gcov_func(Xmvec4, bhspin)

  73.085 μs (3035 allocations: 56.70 KiB)


4×4×4 MArray{Tuple{4, 4, 4}, Float64, 3, 64} with indices SOneTo(4)×SOneTo(4)×SOneTo(4):
[:, :, 1] =
 2.0001e-9     0.001002     -2.49783e-9   -1.60285e-10
 9.98051e-10  -2.00002e-9    2.68288e-16  -7.99824e-11
 1.3848e-12   -2.86865e-12   3.72275e-19   9.95299e-11
 9.37547e-13   9.8227e-10   -3.22668e-8   -7.51337e-14

[:, :, 2] =
  0.001002      2.00194       0.0765128  -8.02902e-5
 -2.00002e-9    0.998972      0.0381799   8.0143e-5
 -2.86865e-12  -0.000689623   0.996017    5.00313e-5
  9.8227e-10    4.57755e-5   -0.0161298   0.97612

[:, :, 3] =
 -2.49783e-9    0.0765128  -55.3659      2.07213e-10
  2.68288e-16   0.0381799  -27.6276     -1.56264e-14
  3.72275e-19   0.996017    -1.82409    -2.16832e-17
 -3.22668e-8   -0.0161298   -0.0259528  17.209

[:, :, 4] =
 -1.60285e-10  -8.02902e-5   2.07213e-10  -0.170971
 -7.99824e-11   8.0143e-5   -1.56264e-14  -0.0853148
  9.95299e-11   5.00313e-5  -2.16832e-17  -0.0532603
 -7.51337e-14   0.97612     17.209        -8.01428e-5

In [48]:
include("../src/main.jl");
@btime gcov_func(Xmvec4, bhspin)

  6.015 μs (159 allocations: 3.44 KiB)


4×4 MMatrix{4, 4, Float64, 16} with indices SOneTo(4)×SOneTo(4):
 -0.998             2.0           0.0           -0.000160277
  2.0               1.002e6  -38410.7          -80.2989
  0.0          -38410.7           2.76815e7      0.0
 -0.000160277     -80.2989        0.0        85481.3

In [42]:
include("../src/main.jl");
@btime gcov_func_fd(Xmvec4, bhspin)

  31.468 μs (1931 allocations: 32.19 KiB)


4×4 MMatrix{4, 4, Float64, 16} with indices SOneTo(4)×SOneTo(4):
 -0.998             2.0             0.0           -0.000160277
  2.0               1.00205e6  -38410.7          -80.2989
  0.0          -38410.7             2.76815e7      0.0
 -0.000160277     -80.2989          0.0        85481.3